# Notebook 46 — Point-in-time provenance, leakage and source reconciliation

**Purpose:** Follow up Notebook 45's unresolved evidence: 238 absolute returns >20%, seven >50%, 220 current-stock universe with no verified historical membership, and potentially globally fitted feature transforms. No new model optimization. **The final historical holdout remains closed.** All database access is optional and read-only. Costs remain zero.

**Interpretation rules:** A matching local raw PostgreSQL record is *not* independent vendor verification. A static AST finding is a *review flag*, not proof of leakage. An internally consistent label is not proof of a correct adjusted price. Do not promote a strategy on these tests alone.

In [1]:
from pathlib import Path
import sys, json, time
import pandas as pd
import numpy as np
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
assert (ROOT/'src').exists(), 'Open notebook from the stock_ml_dynamic repo'
sys.path.insert(0,str(ROOT/'src'))
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.provenance_validation_46 import *
from stock_ml_dynamic.point_in_time_audit_45 import master_integrity,source_inventory
MASTER_DIR=ROOT/'artifacts/master_tables/1d'
master,latest,bundle,manifest=load_master(MASTER_DIR)
OUT=ROOT/'artifacts/provenance_validation/1d';OUT.mkdir(parents=True,exist_ok=True)
print('Master:',master.shape,'latest:',latest.shape,'symbols:',master.symbol.nunique())
print('Source manifest cutoff:',manifest.get('train_cutoff'))

Master: (431299, 27) latest: (220, 27) symbols: 220
Source manifest cutoff: 2025-04-10 00:00:00+00:00


## Experiments 1–8: Source fingerprinting and reproducibility

Record exact SHA-256 checksums of master artifacts and the implementation files. This supports repeatability, not historical point-in-time provenance.

In [2]:
files=list(MASTER_DIR.glob('*'))+list((ROOT/'src/stock_ml_dynamic').glob('*.py'))
hashes=fingerprint(files);display(hashes.loc[hashes.path.str.contains('master_labeled|manifest|master_table.py|candle_table.py')]);hashes.to_csv(OUT/'file_fingerprints.csv',index=False)
print('Fingerprint count:',len(hashes))

,path,bytes,sha256
0,/home/oem/Documents/github/stock_ml_dynamic/ar...,52471510,7662edb08d14541a196cd2c05e88dc5fed9ac5b4ce56a2...
3,/home/oem/Documents/github/stock_ml_dynamic/ar...,989,5969c89546132ba53b04290ac81841b8e6adf5faf34de1...
12,/home/oem/Documents/github/stock_ml_dynamic/sr...,8331,13d966c228dd956e1946ce06a74625ca38762a0d0b5894...
13,/home/oem/Documents/github/stock_ml_dynamic/sr...,9889,1aca27c203ba21702e565d85a275bf431b0e98ba09ffca...


Fingerprint count: 59


## Experiments 9–17: Target alignment and chronology

Verify each available adjacent master row matches the stated next-bar timestamp. The last labeled row per stock may legitimately point to the withheld latest inference row, so it is not compared against a nonexistent next master row.

In [3]:
alignment=master_alignment(master);display(alignment)
assert alignment.loc[alignment.check.eq('duplicate_symbol_time'),'failures'].iloc[0]==0
alignment.to_csv(OUT/'alignment.csv',index=False)
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.walk_forward import expanding_folds
eligible,eligibility_summary=eligible_universe(master)
development,holdout_boundary=development_holdout_split(eligible,.20)
print('Eligible rows:',len(eligible),'development rows:',len(development))
print('Holdout boundary:',holdout_boundary,'(holdout observations not inspected)')
# development_holdout_split returns (development_frame, boundary_timestamp), NOT holdout rows.
# Do not inspect closed-holdout rows, features, labels or performance.
folds=expanding_folds(development,n_folds=3)
chronology=boundary_checks(development,folds);display(chronology)
chronology.to_csv(OUT/'chronology.csv',index=False)
print('Boundary flags above are a conservative raw-time audit; fold_data performs the actual label purge.')

,check,failures,scope
0,duplicate_symbol_time,0,master
1,nonfuture_next_timestamp,0,master
2,next_timestamp_mismatch_internal,0,431079 rows with a following master row
3,nonfinite_next_return,0,master


Eligible rows: 431269 development rows: 343669
Holdout boundary: 2025-04-09 00:00:00+00:00 (holdout observations not inspected)


,fold,train_end,training_rows,validation_rows,train_labels_after_boundary,train_max_label,first_validation
0,1,2022-05-26 00:00:00+00:00,187122,156547,217,2022-05-27 00:00:00+00:00,2022-05-27 00:00:00+00:00
1,2,2023-05-10 00:00:00+00:00,238985,104684,217,2023-05-11 00:00:00+00:00,2023-05-11 00:00:00+00:00
2,3,2024-04-24 00:00:00+00:00,291109,52560,219,2024-04-25 00:00:00+00:00,2024-04-25 00:00:00+00:00


Boundary flags above are a conservative raw-time audit; fold_data performs the actual label purge.


## Experiments 18–27: Source inventory and corporate-action candidates

Identify short-history symbols, outlier events, and date clustering. These are candidates for *independent* split/dividend/news verification; this notebook does not claim to perform that external verification.

In [4]:
coverage=universe_coverage(master);events=event_candidates(master,.20)
display(coverage.head(25));print('Extreme events:',len(events),'above 50%:',int((events.next_return.abs()>.5).sum()))
display(events.head(30))
coverage.to_csv(OUT/'coverage.csv',index=False);events.to_csv(OUT/'event_candidates.csv',index=False)
by_symbol=events.groupby('symbol').agg(events=('next_return','size'),largest_abs=('next_return',lambda x:x.abs().max())).sort_values('events',ascending=False)
by_date=events.groupby('next_bar_timestamp').size().sort_values(ascending=False).head(25)
display(by_symbol.head(20));display(by_date)

,symbol,first_seen,last_seen,bars
210,VYLR,2026-10-01,2026-11-11,30
85,GEV,2024-03-27,2026-11-11,660
167,RDDT,2024-03-21,2026-11-11,664
37,CEG,2022-01-19,2026-11-11,1209
18,APP,2021-04-15,2026-11-11,1402
165,RBLX,2021-03-10,2026-11-11,1427
3,ABNB,2020-12-10,2026-11-11,1487
53,DASH,2020-12-09,2026-11-11,1488
160,PLTR,2020-09-30,2026-11-11,1537
45,CRWD,2019-06-12,2026-11-11,1866


Extreme events: 238 above 50%: 7


,symbol,bar_timestamp,next_bar_timestamp,adj_close,next_adj_close,volume,next_return
0,MRNA,2026-08-18 00:00:00+00:00,2026-08-19 00:00:00+00:00,62.959999,174.380005,4305000,1.769695
1,PCG,2019-01-23 00:00:00+00:00,2019-01-24 00:00:00+00:00,7.820749,13.654498,21631000,0.745932
2,ECHO,2025-08-25 00:00:00+00:00,2025-08-26 00:00:00+00:00,29.879999,50.869999,2493700,0.702477
3,CVNA,2023-06-07 00:00:00+00:00,2023-06-08 00:00:00+00:00,3.106000,4.846000,110955000,0.560206
4,TRGP,2020-03-06 00:00:00+00:00,2020-03-09 00:00:00+00:00,24.265570,11.427291,4463000,-0.529074
5,PCG,2019-01-11 00:00:00+00:00,2019-01-14 00:00:00+00:00,17.217392,8.202486,19750700,-0.523593
6,OXY,2020-03-06 00:00:00+00:00,2020-03-09 00:00:00+00:00,24.176077,11.601182,38969100,-0.520138
7,ECHO,2025-06-13 00:00:00+00:00,2025-06-16 00:00:00+00:00,16.840000,25.110001,5896000,0.491093
8,APP,2024-11-06 00:00:00+00:00,2024-11-07 00:00:00+00:00,168.550003,246.529999,6099600,0.462652
9,CVNA,2023-01-11 00:00:00+00:00,2023-01-12 00:00:00+00:00,1.100000,1.606000,193428500,0.460000


,events,largest_abs
symbol,,
CVNA,33,0.560206
PCG,14,0.745932
PLTR,11,0.308014
APP,9,0.462652
MRNA,8,1.769695
RBLX,8,0.422338
RCL,8,0.317782
SKYD,7,0.367365
TRGP,6,0.529074


next_bar_timestamp
2020-03-16 00:00:00+00:00    16
2020-03-24 00:00:00+00:00    13
2020-03-18 00:00:00+00:00    12
2020-03-09 00:00:00+00:00    12
2020-03-19 00:00:00+00:00    10
2020-03-12 00:00:00+00:00    10
2020-11-09 00:00:00+00:00    10
2020-03-25 00:00:00+00:00     4
2025-01-27 00:00:00+00:00     4
2025-04-09 00:00:00+00:00     4
2020-03-13 00:00:00+00:00     2
2022-05-12 00:00:00+00:00     2
2026-07-31 00:00:00+00:00     2
2020-03-17 00:00:00+00:00     2
2020-06-05 00:00:00+00:00     2
2024-10-30 00:00:00+00:00     2
2026-10-30 00:00:00+00:00     2
2026-02-27 00:00:00+00:00     2
2020-02-26 00:00:00+00:00     1
2020-01-31 00:00:00+00:00     1
2019-11-01 00:00:00+00:00     1
2019-10-30 00:00:00+00:00     1
2019-10-29 00:00:00+00:00     1
2019-10-28 00:00:00+00:00     1
2019-10-25 00:00:00+00:00     1
dtype: int64

## Experiments 28–39: Static feature provenance and look-ahead review

Inventory `shift`, fitting, rolling and aggregation operations with line numbers. Pay special attention to global clustering and historical market aggregation: even if the transforms use only historical rows, a fold can leak if its fitted state incorporates later validation observations. Manual review is mandatory.

In [5]:
calls=code_inventory(ROOT/'src/stock_ml_dynamic');display(calls.loc[calls.review.eq('REVIEW')].head(90));calls.to_csv(OUT/'code_call_inventory.csv',index=False)
print('Review calls:',int(calls.review.eq('REVIEW').sum()))
print('Master manifest fit cutoff:',manifest.get('train_cutoff'))
print('Important: a global master cluster fit before the manifest cutoff can still overlap early validation folds.')

,file,line,call,expression,review
1,master_table.py,108,"df.groupby(['symbol', 'interval_code'], sort=F...","df.groupby(['symbol','interval_code'], sort=Fa...",REVIEW
2,master_table.py,115,DirectionQuantiles().fit,DirectionQuantiles().fit(train_returns),REVIEW
3,master_table.py,117,CandleClusters().fit,"CandleClusters().fit(train_features, k_values=...",REVIEW
6,master_table.py,133,"master.loc[fit_rows, FEATURE_COLUMNS].replace(...","master.loc[fit_rows, FEATURE_COLUMNS].replace(...",REVIEW
7,master_table.py,136,preprocessor.fit,preprocessor.fit(valid_train),REVIEW
11,master_table.py,198,grouped.shift,grouped.shift(-horizon),REVIEW
12,master_table.py,133,"master.loc[fit_rows, FEATURE_COLUMNS].replace","master.loc[fit_rows, FEATURE_COLUMNS].replace(...",REVIEW
14,candle_table.py,40,(h - l).shift,(h-l).shift(1),REVIEW
15,candle_table.py,41,((o - c.shift(1)) / prev_range).where,((o-c.shift(1))/prev_range).where(prev_range.n...,REVIEW
16,candle_table.py,43,adj.shift(1).rolling,"adj.shift(1).rolling(z_window, min_periods=z_w...",REVIEW


Review calls: 59
Master manifest fit cutoff: 2025-04-10 00:00:00+00:00
Important: a global master cluster fit before the manifest cutoff can still overlap early validation folds.


## Experiments 40–49: Optional read-only PostgreSQL reconciliation

The following check compares the current master table to raw price records **already stored in your local database**. It is deliberately disabled by default. Set `RUN_LOCAL_DB_AUDIT=True` to execute. It performs a SELECT only; no table modifications. The comparison is not an independent vendor or corporate-action audit.

In [6]:
RUN_LOCAL_DB_AUDIT=False
raw_comparison=pd.DataFrame();raw_available=False
if RUN_LOCAL_DB_AUDIT:
    from sqlalchemy import create_engine
    from stock_ml_dynamic.master_table import fetch_universe_prices
    import os
    url=os.environ.get('DATABASE_URL')
    if not url: raise RuntimeError('DATABASE_URL not configured; no database query performed')
    engine=create_engine(url)
    with engine.connect() as conn:
        raw=fetch_universe_prices(conn,interval='1d',bars_per_symbol=2000,symbols=master.symbol.unique())
    raw_comparison=compare_raw_to_master(master,raw);raw_available=True
    display(raw_comparison)
    raw_comparison.to_csv(OUT/'raw_source_comparison.csv',index=False)
else:
    print('Local raw DB comparison NOT RUN (opt-in). No external source verification performed.')

Local raw DB comparison NOT RUN (opt-in). No external source verification performed.


## Experiments 50–60: Audit closure and preregistered research gate

The gate defaults to UNVERIFIED for historical universe, adjusted-price corporate actions and feature availability. Do not mark these VERIFIED without independent evidence. This notebook does not read the final holdout or run a new backtest.

In [7]:
gate=decision_gate(alignment,source_available=raw_available)
display(gate)
gate.to_csv(OUT/'evidence_gate.csv',index=False)
report={'notebook':46,'master_rows':int(len(master)),'symbols':int(master.symbol.nunique()),'extreme_over_20pct':int(len(events)),'extreme_over_50pct':int((events.next_return.abs()>.5).sum()),'internal_alignment_failures':int(alignment.failures.sum()),'raw_db_comparison_run':bool(raw_available),'historical_universe':'UNVERIFIED','adjusted_price_vendor_verification':'UNVERIFIED','feature_timing':'UNVERIFIED','final_holdout':'CLOSED','prospective_ledger':'UNTOUCHED','research_decision':'NO MODEL PROMOTION'}
(OUT/'research_gate.json').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))

,check,status,evidence
0,internal_alignment,PASS,0 violations
1,raw_source_comparison,UNVERIFIED,local raw PostgreSQL comparison only; vendor c...
2,historical_universe,UNVERIFIED,point-in-time membership and delistings required
3,corporate_actions,UNVERIFIED,independent split/dividend source required
4,feature_timing,UNVERIFIED,manual decision-time audit required
5,final_holdout,CLOSED,not read


{
  "notebook": 46,
  "master_rows": 431299,
  "symbols": 220,
  "extreme_over_20pct": 238,
  "extreme_over_50pct": 7,
  "internal_alignment_failures": 0,
  "raw_db_comparison_run": false,
  "historical_universe": "UNVERIFIED",
  "adjusted_price_vendor_verification": "UNVERIFIED",
  "feature_timing": "UNVERIFIED",
  "final_holdout": "CLOSED",
  "prospective_ledger": "UNTOUCHED",
  "research_decision": "NO MODEL PROMOTION"
}


## Next action

Review the exported `event_candidates.csv` against an independent adjusted-price/corporate-action source; obtain historical universe membership and delisting records; then audit fold-local feature construction. Only after those evidence gaps are resolved should we design a one-time, preregistered independent holdout evaluation. **No model selection based on final holdout.**